In [10]:
import cv2
import os
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

# --- CONFIGURATION ---
RADIUS = 1
NEIGHBORS = 8
GRID_X = 8
GRID_Y = 8

# Path to your dataset
DATA_PATH = r'..\Dataset\training\Cleaned_Training'

# Path to save test models
MODELS_TEST_PATH = r'..\models\models_test'
os.makedirs(MODELS_TEST_PATH, exist_ok=True)
print(f"Models will be saved to: {MODELS_TEST_PATH}")

Models will be saved to: ..\models\models_test


In [11]:
def rotate_image(image, angle):
    (h, w) = image.shape[:2]
    center = (w // 2, h // 2)
    M = cv2.getRotationMatrix2D(center, angle, 1.0)
    rotated = cv2.warpAffine(image, M, (w, h))
    return rotated

def extract_hog_features(image):
    """
    Extracts Histogram of Oriented Gradients (HOG) features.
    """
    features = hog(image, 
                   orientations=9, 
                   pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), 
                   block_norm='L2-Hys', 
                   visualize=False)
    return features

In [12]:
# ==========================================
# STACKED RECOGNIZER CLASS (Copied from Attendance.ipynb)
# ==========================================
class StackedFaceRecognizer:
    def __init__(self, models_dict, train_feats, train_lbls, id_map=None):
        self.lbph = models_dict.get('LBPH')
        self.svm = models_dict.get('SVM')
        self.xgb = models_dict.get('XGB')
        self.le = models_dict.get('LE')
        self.X_train = np.array(train_feats)
        self.y_train = np.array(train_lbls)
        self.id_map = id_map if id_map else {}
        
        # Internal memory to prevent duplicate counting
        self.session_history = {'SVM': set(), 'XGB': set(), 'LBPH': set()}

    def _reset_session(self):
        """Clears the short-term memory for a new frame/image."""
        self.session_history['SVM'].clear()
        self.session_history['XGB'].clear()
        self.session_history['LBPH'].clear()

    def _extract_hog(self, image):
        return hog(image, orientations=9, pixels_per_cell=(8, 8), 
                   cells_per_block=(2, 2), block_norm='L2-Hys', visualize=False)

    def _predict_single(self, face_img, preds):
        """INTERNAL: Calculates the final vote for ONE face."""
        
        # --- CONFIGURATION ---
        SVM_THRESHOLD = 0.4 
        CONF_THRESHOLD = 0.5   # 50% minimum for SVM/XGB
        LBPH_THRESHOLD = 75    # 75 maximum distance for LBPH
        # ---------------------

        svm_p = preds.get('SVM')
        xgb_p = preds.get('XGB')
        lbph_p = preds.get('LBPH')
        
        valid_votes = []
        valid_confs = []
        
        # 1. Collect Valid Votes (STRICT ENTRY REQUIREMENTS)
        if (svm_p and 
            svm_p['id'] not in self.session_history['SVM'] and 
            svm_p['conf'] >= SVM_THRESHOLD):
            valid_votes.append(svm_p['id'])
            valid_confs.append(svm_p['conf'])
            self.session_history['SVM'].add(svm_p['id'])
            
        if (xgb_p and 
            xgb_p['id'] not in self.session_history['XGB'] and 
            xgb_p['conf'] >= CONF_THRESHOLD):
            valid_votes.append(xgb_p['id'])
            valid_confs.append(xgb_p['conf'])
            self.session_history['XGB'].add(xgb_p['id'])

        if (lbph_p and 
            lbph_p['id'] not in self.session_history['LBPH'] and 
            lbph_p['conf'] <= LBPH_THRESHOLD):
            valid_votes.append(lbph_p['id'])
            valid_confs.append(1.0 if lbph_p['conf'] < 50 else 0.6) 
            self.session_history['LBPH'].add(lbph_p['id'])
        
        # If everyone was filtered out, give up
        if not valid_votes: 
            return None, 0.0, "Weak/No Votes"
        
        # 2. Consensus Logic
        top_vote, count = Counter(valid_votes).most_common(1)[0]
        
        if count >= 2: 
            return top_vote, 1.0, f"Voting ({count}/3)"
        
        if len(set(valid_votes)) == 1:
            idx = valid_votes.index(top_vote)
            return top_vote, valid_confs[idx], "Single Vote"

        # 3. Restricted KNN (Tie Breaker)
        unique_cands = list(set(valid_votes))
        hog_vec = self._extract_hog(face_img)
        mask = np.isin(self.y_train, unique_cands)
        
        if np.sum(mask) == 0: 
            return None, 0.0, "Err"
        
        n_neighbors = min(5, len(self.X_train[mask]))
        mini_knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric='euclidean', algorithm='brute')
        mini_knn.fit(self.X_train[mask], self.y_train[mask])
        
        final_id = mini_knn.predict([hog_vec])[0]
        final_conf = np.max(mini_knn.predict_proba([hog_vec])[0])
        
        return final_id, final_conf, "Restricted KNN"

In [13]:
def load_and_augment_data(path):
    image_paths = [os.path.join(path, f) for f in os.listdir(path)]
    
    # Lists for LBPH (Raw Images)
    lbph_faces = []
    lbph_ids = []
    
    # Lists for SVM & KNN (HOG Feature Vectors)
    hog_features = []
    hog_labels = []
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

    print(f"Processing {len(image_paths)} source images...")
    
    for img_path in image_paths:
        try:
            img = Image.open(img_path).convert('L')
            img_np = np.array(img, 'uint8')
            
            # CAUTION: Ensure filenames are "User.ID.jpg"
            user_id = int(os.path.split(img_path)[-1].split(".")[1])
            
            # Base Preprocessing (200x200 standard)
            face_resized = cv2.resize(img_np, (200, 200), interpolation=cv2.INTER_CUBIC)
            face_smooth = cv2.bilateralFilter(face_resized, 5, 75, 75)
            enhanced = clahe.apply(face_smooth)
            
            # --- AUGMENTATION STACK ---
            aug_imgs = [enhanced]
            aug_imgs.append(cv2.flip(enhanced, 1))                        # Flip
            aug_imgs.append(rotate_image(enhanced, -10))                  # Rotate Left
            aug_imgs.append(rotate_image(enhanced, 10))                   # Rotate Right
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=-40)) # Darker
            aug_imgs.append(cv2.convertScaleAbs(enhanced, alpha=1, beta=40))  # Brighter
            
            # Add to datasets
            for face in aug_imgs:
                # 1. For LBPH: Add the raw image
                lbph_faces.append(face)
                lbph_ids.append(user_id)
                
                # 2. For SVM & KNN: Extract HOG features
                feat = extract_hog_features(face)
                hog_features.append(feat)
                hog_labels.append(user_id)
                
        except Exception as e:
            print(f"Skipping {img_path}: {e}")
            
    return np.array(lbph_faces), np.array(lbph_ids), np.array(hog_features), np.array(hog_labels)

# --- EXECUTE LOAD ---
faces, ids, hog_feats, hog_lbls = load_and_augment_data(DATA_PATH)

if len(faces) > 0:
    print(f"✅ Data Loaded successfully.")
    print(f"Total Samples: {len(faces)}")
else:
    print("⚠️ No data found. Check your path.")

Processing 139 source images...
✅ Data Loaded successfully.
Total Samples: 834


In [14]:
# --- DATA SPLITTING ---
# Splitting HOG features (SVM/XGBoost) - 75% Train, 25% Test
X_train_hog, X_test_hog, y_train_hog, y_test_hog = train_test_split(
    hog_feats, hog_lbls, test_size=0.25, random_state=42, stratify=hog_lbls
)

# Splitting Raw Faces (LBPH) - 75% Train, 25% Test
faces_train, faces_test, ids_train, ids_test = train_test_split(
    faces, ids, test_size=0.25, random_state=42, stratify=ids
)

print(f"Training Set Size: {len(faces_train)}")
print(f"Testing Set Size: {len(faces_test)}")

Training Set Size: 625
Testing Set Size: 209


In [15]:
def print_metrics(model_name, y_true, y_pred):
    print(f"\n--- {model_name} EVALUATION ---")
    
    # Note: Stacker might return mixed types or None if unclassified, handling below
    
    # Overall Metrics
    acc = accuracy_score(y_true, y_pred)
    print(f"Overall Accuracy:  {acc:.4f}")
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    classes = sorted(list(set(y_true) | set(y_pred)))
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
    plt.title(f'{model_name} Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    print(f"\nDetailed Metrics per Class for {model_name}:")
    print(f"{'Class':<10} {'TP':<6} {'TN':<6} {'FP':<6} {'FN':<6} {'Accuracy':<10} {'Precision':<10} {'Sensitivity':<12} {'Specificity':<12}")
    print("-" * 90)
    
    FP = cm.sum(axis=0) - np.diag(cm)  
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    for i, cls_label in enumerate(classes):
        tp_val = TP[i]
        tn_val = TN[i]
        fp_val = FP[i]
        fn_val = FN[i]
        
        sensitivity = tp_val / (tp_val + fn_val) if (tp_val + fn_val) > 0 else 0
        specificity = tn_val / (tn_val + fp_val) if (tn_val + fp_val) > 0 else 0
        precision = tp_val / (tp_val + fp_val) if (tp_val + fp_val) > 0 else 0
        accuracy_cls = (tp_val + tn_val) / (tp_val + tn_val + fp_val + fn_val)
        
        print(f"{cls_label:<10} {tp_val:<6} {tn_val:<6} {fp_val:<6} {fn_val:<6} {accuracy_cls:<10.4f} {precision:<10.4f} {sensitivity:<12.4f} {specificity:<12.4f}")
    
    print("-" * 90)

In [16]:
# --- LBPH EVALUATION ---
if len(faces_train) > 0:
    print("Training LBPH Model...")
    lbph = cv2.face.LBPHFaceRecognizer_create(radius=RADIUS, neighbors=NEIGHBORS, grid_x=GRID_X, grid_y=GRID_Y)
    lbph.train(faces_train, ids_train)
    
    # Save Model
    save_path = os.path.join(MODELS_TEST_PATH, 'trainer_test.yml')
    lbph.save(save_path)
    print(f"✅ Saved '{save_path}'")
    
    print("Evaluating LBPH...")
    y_pred_lbph = []
    for face in faces_test:
        label, confidence = lbph.predict(face)
        y_pred_lbph.append(label)
        
    print_metrics("LBPH", ids_test, y_pred_lbph)

Training LBPH Model...
✅ Saved '..\models\models_test\trainer_test.yml'
Evaluating LBPH...

--- LBPH EVALUATION ---
Overall Accuracy:  0.9522


<Figure size 1000x800 with 2 Axes>


Detailed Metrics per Class for LBPH:
Class      TP     TN     FP     FN     Accuracy   Precision  Sensitivity  Specificity 
------------------------------------------------------------------------------------------
1          37     169    1      2      0.9856     0.9737     0.9487       0.9941      
2          37     171    0      1      0.9952     1.0000     0.9737       1.0000      
3          27     177    2      3      0.9761     0.9310     0.9000       0.9888      
4          49     155    4      1      0.9761     0.9245     0.9800       0.9748      
5          27     180    1      1      0.9904     0.9643     0.9643       0.9945      
6          22     183    2      2      0.9809     0.9167     0.9167       0.9892      
------------------------------------------------------------------------------------------


In [17]:
# --- SVM EVALUATION ---
if len(X_train_hog) > 0:
    print("Training SVM Model...")
    svm = SVC(kernel='linear', C=10.0, gamma='scale', probability=True, random_state=42)
    svm.fit(X_train_hog, y_train_hog)
    
    # Save Model
    save_path = os.path.join(MODELS_TEST_PATH, 'svm_face_model_test.pkl')
    joblib.dump(svm, save_path)
    print(f"✅ Saved '{save_path}'")
    
    print("Evaluating SVM...")
    y_pred_svm = svm.predict(X_test_hog)
    print_metrics("SVM", y_test_hog, y_pred_svm)

Training SVM Model...
✅ Saved '..\models\models_test\svm_face_model_test.pkl'
Evaluating SVM...

--- SVM EVALUATION ---
Overall Accuracy:  0.9474


<Figure size 1000x800 with 2 Axes>


Detailed Metrics per Class for SVM:
Class      TP     TN     FP     FN     Accuracy   Precision  Sensitivity  Specificity 
------------------------------------------------------------------------------------------
1          38     165    5      1      0.9713     0.8837     0.9744       0.9706      
2          38     171    0      0      1.0000     1.0000     1.0000       1.0000      
3          27     179    0      3      0.9856     1.0000     0.9000       1.0000      
4          49     154    5      1      0.9713     0.9074     0.9800       0.9686      
5          26     181    0      2      0.9904     1.0000     0.9286       1.0000      
6          20     184    1      4      0.9761     0.9524     0.8333       0.9946      
------------------------------------------------------------------------------------------


In [18]:
# --- XGBoost EVALUATION ---
if len(X_train_hog) > 0:
    print("Training XGBoost Model...")
    
    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train_hog)
    y_test_enc = le.transform(y_test_hog)
    
    xgb_model = XGBClassifier(
        objective='multi:softprob',
        num_class=len(le.classes_),
        n_estimators=200,
        max_depth=6,
        learning_rate=0.5,
        subsample=0.8,
        colsample_bytree=1,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1
    )
    
    xgb_model.fit(X_train_hog, y_train_enc)
    
    # Save Model
    save_path_xgb = os.path.join(MODELS_TEST_PATH, 'xgb_face_model_test.pkl')
    save_path_le = os.path.join(MODELS_TEST_PATH, 'label_encoder_test.pkl')
    joblib.dump(xgb_model, save_path_xgb)
    joblib.dump(le, save_path_le)
    print(f"✅ Saved '{save_path_xgb}' and '{save_path_le}'")
    
    print("Evaluating XGBoost...")
    y_pred_xgb_enc = xgb_model.predict(X_test_hog)
    y_pred_xgb = le.inverse_transform(y_pred_xgb_enc)
    
    print_metrics("XGBoost", y_test_hog, y_pred_xgb)

Training XGBoost Model...
✅ Saved '..\models\models_test\xgb_face_model_test.pkl' and '..\models\models_test\label_encoder_test.pkl'
Evaluating XGBoost...

--- XGBoost EVALUATION ---
Overall Accuracy:  0.8469


<Figure size 1000x800 with 2 Axes>


Detailed Metrics per Class for XGBoost:
Class      TP     TN     FP     FN     Accuracy   Precision  Sensitivity  Specificity 
------------------------------------------------------------------------------------------
1          32     161    9      7      0.9234     0.7805     0.8205       0.9471      
2          36     166    5      2      0.9665     0.8780     0.9474       0.9708      
3          24     177    2      6      0.9617     0.9231     0.8000       0.9888      
4          43     147    12     7      0.9091     0.7818     0.8600       0.9245      
5          23     179    2      5      0.9665     0.9200     0.8214       0.9890      
6          19     183    2      5      0.9665     0.9048     0.7917       0.9892      
------------------------------------------------------------------------------------------


In [20]:
# --- STACKER EVALUATION ---

if 'lbph' in locals() and 'svm' in locals() and 'xgb_model' in locals():
    print("Initializing Stacker with Test Models...")
    
    # Create a dummy ID map (not strictly needed for metrics, but required by init)
    id_map = {uid: f"User {uid}" for uid in np.unique(hog_lbls)}
    
    # Initialize Stacker
    models = {
        'LBPH': lbph,
        'SVM': svm,
        'XGB': xgb_model,
        'LE': le
    }
    stacker = StackedFaceRecognizer(models, X_train_hog, y_train_hog, id_map)
    
    print("Evaluating Stacker on Test Set...")
    y_true_stacker = ids_test
    y_pred_stacker = []
    unclassified_count = 0
    
    # Simulate Frame-by-Frame Prediction for Test Set
    for idx, face_img in enumerate(faces_test):
        hog_vec = X_test_hog[idx]
        
        # 1. Get Individual Predictions
        preds = {}
        
        # SVM
        try:
            prob_svm = svm.predict_proba([hog_vec])[0]
            conf_svm = np.max(prob_svm)
            pid_svm = svm.classes_[np.argmax(prob_svm)]
            preds['SVM'] = {'id': pid_svm, 'conf': conf_svm}
        except: pass
        
        # XGB
        try:
            prob_xgb = xgb_model.predict_proba([hog_vec])[0]
            conf_xgb = np.max(prob_xgb)
            decoded_id = le.inverse_transform([np.argmax(prob_xgb)])[0]
            preds['XGB'] = {'id': decoded_id, 'conf': conf_xgb}
        except: pass
        
        # LBPH
        try:
            pid_lbph, dist_lbph = lbph.predict(face_img)
            preds['LBPH'] = {'id': pid_lbph, 'conf': dist_lbph}
        except: pass
        
        # 2. Stacker Prediction
        stacker._reset_session() # IMPORTANT: Reset memory for independent test samples
        final_id, final_conf, method = stacker._predict_single(face_img, preds)
        
        if final_id is None:
            # Handle unclassified cases (assign a dummy ID that will count as error)
            # We use 99999 as it's likely not a valid ID
            final_id = 99999
            unclassified_count += 1
            
        y_pred_stacker.append(final_id)

    print(f"Stacker Evaluation Complete. Unclassified instances: {unclassified_count}")
    print_metrics("Stacker", y_true_stacker, y_pred_stacker)
    
    # Save Stacker (The class object)
    # NOTE: OpenCV's LBPHFaceRecognizer cannot be pickled. We remove it before saving.
    # It is saved separately as 'trainer_test.yml'.
    stacker.lbph = None 

    stacker_path = os.path.join(MODELS_TEST_PATH, 'stacker_model_test.pkl')
    joblib.dump(stacker, stacker_path)
    print(f"✅ Saved Stacker Object (excluding LBPH) to '{stacker_path}'")
    print("ℹ️  To reload: Load stacker pkl, then load LBPH from 'trainer_test.yml' and assign to stacker.lbph")